# Notebook 3: Semantic search via Embeddings

The Europeana section of the intro notebook found objects through their *metadata*. Now we will find results by comparing **Embeddings**. The 
[CLIP](https://openai.com/research/clip) Model computes embeddings for images *and* short texts in the same space, so a sentence can be compared directly with every image.

## CLIP: text and images in one space

![CLIP contrastive pre-training: a text encoder and an image encoder produce vectors that are compared in a matrix](../misc/assets/img/clip.svg)

<small>Contrastive pre-training of CLIP: matching image-text pairs are pulled together (blue diagonal), all other pairs pushed apart. Figure: OpenAI, ["CLIP: Connecting text and images"](https://openai.com/index/clip/) (Radford et al., 2021).</small>

The image half was applied to the whole collection beforehand. For a query only the **text encoder** runs, which is fast on a laptop.

<div style="margin:80px 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 1 / 7</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:3px solid #BC8F8F;margin:6px 0 0 0">

In [41]:
# Setup: run this cell first.
import numpy as np
from IPython.display import display
from clip_helpers import *
from region_helpers import *
from iiif_helpers import show_mirador_many, url_panel

<div style="margin:80px 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 2 / 7</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:3px solid #BC8F8F;margin:6px 0 0 0">

## The collection

`metadata.csv` and the vectors are in the same order: row *i* describes vector *i*. Four random images (the images are fetched from their institutions now):

In [ ]:
coll = Collection()
print("About 600,000 images from Europeana, up to about 8,000 from each institution.")

rng = np.random.default_rng(1)
coll.show_grid(rng.choice(len(coll.meta), 4, replace=False))
clip_model = Clip()

<div style="margin:80px 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 3 / 7</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:3px solid #BC8F8F;margin:6px 0 0 0">

## Search with Europeana keywords

The Europeana search of the previous notebook looks for **words** in the metadata: title, description, creator, subject. Here a keyword search for *Amor and Psyche*, restricted to open images of the Art collection. Keep the result in mind: it is the comparison for the next section.

In [ ]:
import europeana_helpers as eu

KEYWORD = "Amor and Psyche"
result = eu.search_europeana(query=KEYWORD, rows=12, qf=["TYPE:IMAGE"], theme="art", reusability="open")
# a filter here is a Europeana filter: add it to qf, for example qf=["TYPE:IMAGE", "YEAR:[1800 TO 1899]"]
print(f"{result['totalResults']} records contain these words. The first 12:")
eu.show_results(result["items"])

<div style="margin:80px 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 4 / 7</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:3px solid #BC8F8F;margin:6px 0 0 0">

## Search with a Text Embedding

The query becomes a vector in the same 512-dimensional space. Both are normalised to length 1, so their **dot product** is their cosine similarity: 1 means the same direction, near 0 unrelated. Ranking the whole collection is one matrix–vector product.

In [ ]:
QUERY = "Amor and Psyche"

scores = coll.similarity(clip_model.text(QUERY))
best = coll.top_k(scores, 12)
coll.show_grid(best, scores)

<div style="margin:80px 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 5 / 7</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:3px solid #BC8F8F;margin:6px 0 0 0">

## Restricting the search with a filter

Because the metadata is aligned with the vectors, a **filter** is just a list of true/false values, one per image, applied before ranking: only images marked true can appear in the results. Here: only images dated before 1900.

In [ ]:
QUERY = "a portrait of a person"
keep = (coll.year > 0) & (coll.year < 1900)                          # dated, and before 1900
# keep = np.isin(coll.country, ["Sweden", "Finland", "Norway"])      # alternative: by country
# keep = np.isin(coll.institution, ["Wellcome Collection"])          # alternative: by institution

scores = coll.similarity(clip_model.text(QUERY))
print(f"{keep.sum():,} of {len(keep):,} images pass the filter. Top results for {QUERY!r}:\n")
coll.show_grid(coll.top_k(scores, 12, keep), scores)

<div style="margin:80px 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 6 / 7</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:3px solid #BC8F8F;margin:6px 0 0 0">

## Search with an image

The same space holds image embeddings, so an image can be the query: "more like this". **Upload an image** of your own (a photograph, a scan, a screenshot), then run the cell below it. Without an upload, the cell uses the best match of the text search above.

In [ ]:
import io
import ipywidgets as widgets
from PIL import Image

upload = widgets.FileUpload(accept="image/*", multiple=False, description="Upload an image")
display(upload)

In [ ]:
files = list(upload.value.values()) if isinstance(upload.value, dict) else list(upload.value)

if files:                                  # your own image
    query_image = Image.open(io.BytesIO(bytes(files[0]["content"]))).convert("RGB")
    print("Query: your image")
    display(query_image.copy().resize((200, int(200 * query_image.height / query_image.width))))
    scores = coll.similarity(clip_model.image(query_image))
    coll.show_grid(coll.top_k(scores, 12), scores)
else:                                      # no upload: more like the best match of the text search
    SEED = int(best[0])
    print("Query: collection image", SEED, "-", coll.meta[SEED]["title"][:60])
    display(coll.small_image(SEED))
    scores = coll.similarity(coll.V[SEED].astype(np.float32))
    scores[SEED] = -1                      # do not return the query itself
    coll.show_grid(coll.top_k(scores, 12), scores)

<div style="margin:80px 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 7 / 7</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:3px solid #BC8F8F;margin:6px 0 0 0">

## One picture, five regions

An image is rarely about one thing. We cut it into **regions**, embed each region on its own, and search the collection with it.

A Swedish advertising stamp (*reklammärke*) of 1939 for *Ljusäkta Fiona Färger*, from the Malmö Museum ([record](http://carlotta.malmo.se/carlotta-mmus/web/object/178916), CC BY 4.0). A kneeling figure by candlelight, a snake beside it, a band of coloured stripes below. We search with the whole stamp, the figure, only the face, one foot, the snake, and the motif without the perforated edge (the "teeth").

This record has no IIIF image service and its original file is not available, so the regions are cut from the small version (400 px) on your computer: coarse, which matters for small regions. `frac(x, y, w, h)` gives a region as fractions of the picture (0 to 1), converted to pixels.

The stamp is tall. With `fit="crop"` CLIP reads the central square, as it did for the vectors of the collection; the default `"pad"` would add grey bars and the search would follow the grey (try it).

In [ ]:
ID = "/91658/MM_objekt_178916"      # a record of the collection (Europeana ID)
row = next(i for i, r in enumerate(coll.meta) if r["id"] == ID)
pic = Picture(row, coll)

regions = {
    "kneeling figure":  pic.frac(0.20, 0.19, 0.50, 0.54),
    "face":             pic.frac(0.37, 0.20, 0.22, 0.17),
    "foot":             pic.frac(0.34, 0.59, 0.14, 0.13),
    "snake":            pic.frac(0.69, 0.54, 0.19, 0.20),
    "motif (no teeth)": pic.frac(0.055, 0.04, 0.87, 0.915),
}
pic.show(regions)

The same search with seven queries: the whole stamp and the five regions. Each row is one query, each card one of the 8 most similar images of the collection (the stamp itself is left out).

In [ ]:
exp = Experiment(coll, clip_model, pic)
exp.add("whole stamp", fit="crop")
for name, region in regions.items():
    exp.add(name, region, fit="crop")

exp.run(k=8)
exp.show()